# ECHR structured extraction → auditable case table (Layer 1)

A **structured-extraction layer** over ECHR judgments: one row per case, every cell carrying a
**confidence** value and a **provenance** pointer (section/source), written to
`data/echr_extracted.parquet`. The aggregate questions that RAG abstains on (Bucket-3
content-aggregates) are then answered over *columns*, not over generated prose.

**Scope: ECHR only.** RIS stays retrieval-only — Rechtssätze have no facts/parties/outcome to
tabulate, so no table is built for them. Swiss is out of scope.

Two tiers, by how much extraction effort the field genuinely needs (inspection-driven):
- **Metadata tier (high confidence)** — `respondent_state`, `articles`, `judgment_date`,
  `importance`, `genre`, `outcome`. The corpus already carries these as structured HUDOC
  fields, so we **normalise**, we do not re-extract. `outcome` is recovered from the
  `conclusion`/`violation`/`nonviolation` metadata (operative paragraph only as a fallback).
- **Content tier (lower confidence)** — `alienation_alleged` (+ `alienation_conf`,
  `alienation_evidence`). Genuinely unstructured: a rules/lexicon detector restricted to
  applicant/facts context with negation handling. `alienation_conf` is a transparent function
  of signal strength — **a low-confidence cell is field-level abstention**.

**Labels never appear here.** This notebook runs with *no* hand-labels; calibration
(`extraction_validation.ipynb`) is the only place gold labels are read, offline.

## 1. Configuration

In [ ]:
from pathlib import Path

DATA_DIR   = Path("../data")
ECHR_FILE  = DATA_DIR / "echr_parental_alienation.json"

OUT_TABLE      = DATA_DIR / "echr_extracted.parquet"          # the auditable case table
LABEL_TEMPLATE = DATA_DIR / "echr_label_template.csv"         # stratified hand-label template
CALIB_FILE     = DATA_DIR / "alienation_calibration.joblib"   # written by extraction_validation

# alienation detector / sampling parameters (all transparent, CPU-cheap)
ALLEGED_THRESHOLD = 0.50     # alienation_conf >= this -> alienation_alleged = True
CONF_HIGH         = 0.70     # "confident allegation" band (used by the query layer demos)
SAMPLE_N          = 120      # stratified labelling sample size (confirmed: ~50/50, oversample +ve)
RANDOM_SEED       = 42

DOCTYPE_GENRE = {"HEJUD": "merits", "HEDEC": "admissibility", "HECOM": "communicated"}

print("input :", ECHR_FILE.name)
print("output:", OUT_TABLE.name, "| label template:", LABEL_TEMPLATE.name)
print(f"alleged>= {ALLEGED_THRESHOLD} | high-conf band {CONF_HIGH} | sample N={SAMPLE_N}")

input : echr_parental_alienation.json
output: echr_extracted.parquet | label template: echr_label_template.csv
alleged>= 0.5 | high-conf band 0.7 | sample N=120


## 2. Load + genre helper
`itemid` is the join key; `genre` is derived from the HUDOC `doctype` exactly as in the prior
retrieval task, so the table is **joinable by id** to the RAG corpus. `outcome` is forced to
`null` for communicated cases (no ruling) — never guessed.

In [ ]:
import json, re

def load_json_records(path):
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return data if isinstance(data, list) else [data]


def echr_genre(doctype):
    """Reusable genre helper (copy of the retrieval-task helper): doctype -> genre."""
    return DOCTYPE_GENRE.get((doctype or "").upper(), "other")


records = load_json_records(ECHR_FILE)
if records is None:
    print(f"!! {ECHR_FILE} not found — run the import/scraper notebooks first.")
else:
    from collections import Counter
    print(f"loaded {len(records)} ECHR records")
    print("genre:", dict(Counter(echr_genre(r.get("doctype")) for r in records)))

loaded 1116 ECHR records
genre: {'admissibility': 311, 'merits': 567, 'communicated': 238}


## 3. Section splitter (context restriction)
The same heading anchors used by the RAG notebook, plus `SUBJECT MATTER OF THE CASE` (the
communicated-case facts block). The alienation detector only trusts mentions in the
**applicant/facts** context (HEADER/PROCEDURE/FACTS) and applicant-attributed restatements
under `THE LAW`; quoted statutes (`RELEVANT_LAW`), dissents (`OPINION`) and the operative
paragraph are excluded so the Court's own use of the word doesn't count as an allegation.

In [ ]:
ECHR_ANCHORS = [
    (re.compile(r"PROCEDURE(?=[0-9IVX])"), "PROCEDURE"),
    (re.compile(r"THE FACTS(?=[0-9IVX]|\s)"), "FACTS"),
    (re.compile(r"THE CIRCUMSTANCES OF THE CASE"), "FACTS"),
    (re.compile(r"SUBJECT MATTER OF THE CASE"), "FACTS"),
    (re.compile(r"RELEVANT (?:DOMESTIC|LEGAL|INTERNATIONAL|EUROPEAN|COMPARATIVE|COUNCIL)"
                r"[A-Z ]{0,40}?(?:LAW|FRAMEWORK|MATERIAL|PRACTICE|TEXT)"), "RELEVANT_LAW"),
    (re.compile(r"(?:AS TO )?THE LAW(?![a-z])|THE COURT[^A-Za-z]{0,2}S ASSESSMENT"), "LAW"),
    (re.compile(r"FOR THESE REASONS"), "OPERATIVE"),
    (re.compile(r"(?:JOINT |PARTLY )?(?:DISSENTING|SEPARATE|CONCURRING) OPINION"), "OPINION"),
]
APPLICANT_CTX = {"HEADER", "PROCEDURE", "FACTS", "unparsed"}   # where an allegation lives
_SENT = re.compile(r"(?<=[.!?])\s+")


def echr_sections(full_text):
    full_text = full_text or ""
    marks = sorted((m.start(), lab) for rx, lab in ECHR_ANCHORS for m in rx.finditer(full_text))
    if not marks:
        return [("unparsed", full_text)]
    segs = []
    if marks[0][0] > 0:
        segs.append(("HEADER", full_text[:marks[0][0]]))
    for i, (pos, lab) in enumerate(marks):
        end = marks[i + 1][0] if i + 1 < len(marks) else len(full_text)
        segs.append((lab, full_text[pos:end]))
    return segs


print("section splitter ready | applicant-context sections:", APPLICANT_CTX)

section splitter ready | applicant-context sections: {'FACTS', 'unparsed', 'HEADER', 'PROCEDURE'}


## 4. Metadata tier — normalise the structured HUDOC fields (high confidence)
**Inspection finding:** respondent (ISO-3), article (`;`-lists), importance (1–4) and
judgement date are already clean structured columns — we normalise, not re-extract. `outcome`
is derived metadata-first:
- **communicated → `null`** (no ruling);
- **admissibility → `inadmissible` / `struck out`** from `conclusion`;
- **merits → `violation` / `no violation` / `struck out`** from the structured
  `violation`/`nonviolation` (Article 8) fields, then `conclusion`, then the operative
  `FOR THESE REASONS` paragraph as a last-resort fallback (lower confidence).

Each value records a per-field confidence and a provenance pointer.

In [ ]:
from datetime import datetime

_EN_MONTHS = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july", "august",
     "september", "october", "november", "december"], 1)}
_FT_DATE = re.compile(r"(?:communicated on|strasbourg,?|judgment\s+strasbourg|decision\s+strasbourg)"
                      r"\s+(\d{1,2})\s+([A-Za-z]+)\s+(\d{4})", re.IGNORECASE)


def norm_date(r):
    raw = (r.get("judgementdate") or "").strip()
    if raw:
        try:
            return datetime.strptime(raw.split()[0], "%d/%m/%Y").date().isoformat(), 1.0, "meta:judgementdate"
        except (ValueError, IndexError):
            pass
    m = re.match(r"ECLI:CE:ECHR:(\d{4}):(\d{2})(\d{2})", r.get("ecli", "") or "")
    if m:
        y, mo, d = m.groups()
        if 1 <= int(mo) <= 12 and 1 <= int(d) <= 31:
            return f"{y}-{mo}-{d}", 0.9, "ecli"
    fm = _FT_DATE.search((r.get("full_text", "") or "")[:4000])
    if fm:
        d, mon, y = fm.groups()
        mi = _EN_MONTHS.get(mon.lower())
        if mi:
            return f"{y}-{mi:02d}-{int(d):02d}", 0.7, "text:date-phrase"
    return None, 0.0, "no-date"


def norm_articles(r):
    arts = sorted({p.split("-")[0].strip() for p in (r.get("article", "") or "").split(";")
                   if p.strip() and p.split("-")[0].strip().isdigit()}, key=int)
    return ";".join(arts), (1.0 if arts else 0.0), "meta:article"


def _arts(s):
    return {p.split("-")[0].strip() for p in (s or "").split(";") if p.strip()}

_RX_VIOL = re.compile(r"(?<!no )violation of article", re.IGNORECASE)
_RX_NOVIOL = re.compile(r"no violation of article", re.IGNORECASE)
_RX_STRUCK = re.compile(r"strike out|struck out|strike the application", re.IGNORECASE)


def norm_outcome(r, genre, sect_text):
    conc = r.get("conclusion", "") or ""
    viol, noviol = _arts(r.get("violation", "")), _arts(r.get("nonviolation", ""))
    if genre == "communicated":
        return None, 0.99, "genre=communicated(null:no-ruling)"
    if genre == "admissibility":
        if re.search(r"\bInadmissible\b", conc): return "inadmissible", 0.95, "meta:conclusion"
        if re.search(r"Struck out", conc):        return "struck out", 0.90, "meta:conclusion"
        if re.search(r"\bAdmissible\b", conc):   return "admissible", 0.85, "meta:conclusion"
        return None, 0.40, "admissibility:no-marker"
    # merits
    if re.search(r"Struck out", conc) and not viol and not noviol:
        return "struck out", 0.90, "meta:conclusion"
    if "8" in viol:    return "violation", 0.95, "meta:violation(art8)"
    if "8" in noviol:  return "no violation", 0.95, "meta:nonviolation(art8)"
    if _RX_NOVIOL.search(conc): return "no violation", 0.80, "meta:conclusion"
    if _RX_VIOL.search(conc):   return "violation", 0.80, "meta:conclusion"
    op = sect_text.get("OPERATIVE", "")
    if op:
        if _RX_NOVIOL.search(op): return "no violation", 0.60, "text:operative"
        if _RX_VIOL.search(op):   return "violation", 0.60, "text:operative"
        if _RX_STRUCK.search(op):  return "struck out", 0.60, "text:operative"
    return None, 0.30, "merits:no-marker"


print("metadata normalisers ready (respondent / articles / date / importance / outcome)")

metadata normalisers ready (respondent / articles / date / importance / outcome)


## 5. Content tier — alienation_alleged (rules/lexicon, transparent confidence)
**Definition (confirmed):** `alienation_alleged = True` when **a party (typically the
applicant) asserts that the other parent alienated / turned the child against them**, recorded
in the facts/complaints. This is the **allegation, not the holding** — whether the Court
accepted it is irrelevant; court/third-party narration with no party attribution is `False`.

`alienation_conf` = **P(alleged = True)**, a transparent sum of signals:
- **party attribution** (applicant/parent + a saying-verb in the same sentence) — the strong
  signal, base 0.65; attributed-but-negated 0.40; bare facts mention 0.30;
- **multi-mention** (+0.12), **explicit cluster phrase** (+0.10), **negated-only** (−0.25).

`alienation_alleged = conf >= ALLEGED_THRESHOLD`. The mid band (≈0.4–0.7) is exactly the
**field-level abstention zone** the query layer can exclude. No transformers; pure regex.

In [ ]:
CLUSTER = re.compile(
    r"parental alienation|alienat\w*|estrange\w*|"
    r"turn(?:ed|ing|s)?\s+(?:the\s+)?child(?:ren)?\s+against|"
    r"set(?:ting|s)?\s+(?:the\s+)?child(?:ren)?\s+against|"
    r"manipulat\w*\s+(?:the\s+)?child(?:ren)?|"
    r"poison\w*\s+(?:the\s+)?(?:child|mind)", re.IGNORECASE)
ATTRIB = re.compile(r"\b(applicant|mother|father|parent|she|he)\b[^.]{0,80}?\b"
                    r"(alleg\w+|submitt\w+|complain\w+|claim\w+|argu\w+|maintain\w+|"
                    r"contend\w+|stat\w+|assert\w+|accus\w+)\b", re.IGNORECASE)
NEG = re.compile(r"\b(no|not|never|without|denied|rejected|dismissed|unfounded|no evidence)\b",
                 re.IGNORECASE)


def _cluster_sents(sections, labels):
    out = []
    for lab, t in sections:
        if lab in labels:
            for s in _SENT.split(t):
                if CLUSTER.search(s):
                    out.append((lab, s.strip()))
    return out


def alienation_extract(sections):
    """conf = P(alleged=True); alleged = conf>=ALLEGED_THRESHOLD. Returns dict with provenance."""
    all_hits = CLUSTER.findall("\n".join(t for _, t in sections))
    if not all_hits:
        return dict(alleged=False, conf=0.02, evidence=None, prov="no-cluster")
    ctx = _cluster_sents(sections, APPLICANT_CTX)
    law_attr = [(l, s) for l, s in _cluster_sents(sections, {"LAW"}) if ATTRIB.search(s)]
    pool = ctx + law_attr
    if not pool:                          # cluster only in quoted law / dissent / operative
        return dict(alleged=False, conf=0.12, evidence=None,
                    prov=f"cluster-out-of-context(n={len(all_hits)})")
    attributed = [(l, s) for l, s in pool if ATTRIB.search(s)]
    pos_attr = [(l, s) for l, s in attributed if not NEG.search(s)]
    negated_only = all(NEG.search(s) for _, s in pool)
    n = len(pool)

    if pos_attr:       score = 0.65       # party-attributed assertion (strong)
    elif attributed:   score = 0.40       # attributed but negated/rejected
    else:              score = 0.30       # bare facts mention, no attribution verb
    if n >= 2:         score += 0.12
    if re.search(r"parental alienation|turn\w*\s+(?:the\s+)?child(?:ren)?\s+against",
                 " ".join(s for _, s in pool), re.I):
        score += 0.10
    if negated_only and not pos_attr:
        score -= 0.25
    score = max(0.0, min(1.0, round(score, 3)))

    src = (pos_attr or attributed or pool)[0]
    return dict(alleged=score >= ALLEGED_THRESHOLD, conf=score, evidence=src[1][:300],
                prov=f"section={src[0]};mentions={n};attributed={len(pos_attr)};neg_only={negated_only}")


print("alienation extractor ready (lexicon + attribution + negation)")

alienation extractor ready (lexicon + attribution + negation)


## 6. Build the table — one row per case, every cell with confidence + provenance
`confidence` and `provenance` are stored as JSON maps (field → value) so **any cell is
traceable back to text**; the two queried scalar confidences (`outcome_conf`,
`alienation_conf`) are also promoted to top-level columns. If a calibration mapping exists
(written later by `extraction_validation.ipynb`), a calibrated `alienation_conf_cal` column is
added — labelled once, then applied to all rows without re-labelling.

In [ ]:
import pandas as pd

def build_rows(records):
    rows = []
    for r in records:
        iid = r.get("itemid", "") or ""
        genre = echr_genre(r.get("doctype"))
        secs = echr_sections(r.get("full_text", "") or "")
        sect_text = {lab: t for lab, t in secs}
        date, date_c, date_p = norm_date(r)
        arts, art_c, art_p = norm_articles(r)
        oc, oc_c, oc_p = norm_outcome(r, genre, sect_text)
        al = alienation_extract(secs)
        imp = r.get("importance") or None
        conf = {"respondent_state": 1.0, "articles": art_c, "judgment_date": date_c,
                "importance": 1.0 if imp else 0.0, "genre": 1.0,
                "outcome": oc_c, "alienation_alleged": al["conf"]}
        prov = {"respondent_state": "meta:respondent", "articles": art_p,
                "judgment_date": date_p, "importance": "meta:importance",
                "genre": "meta:doctype", "outcome": oc_p, "alienation_alleged": al["prov"]}
        rows.append({
            "id": iid,
            "title": r.get("docname", "") or iid,
            "respondent_state": r.get("respondent", "") or None,
            "articles": arts or None,
            "judgment_date": date,
            "importance": int(imp) if imp and str(imp).isdigit() else None,
            "genre": genre,
            "outcome": oc,
            "outcome_conf": oc_c,
            "alienation_alleged": bool(al["alleged"]),
            "alienation_conf": al["conf"],
            "alienation_evidence": al["evidence"],
            "url": f"https://hudoc.echr.coe.int/eng?i={iid}" if iid else "",
            "confidence": json.dumps(conf),
            "provenance": json.dumps(prov),
        })
    return pd.DataFrame(rows)


def apply_calibration(df):
    """If a calibration model exists, map raw -> calibrated alienation_conf (runtime meaning)."""
    if not CALIB_FILE.exists():
        df["alienation_conf_cal"] = pd.NA
        return df, False
    import joblib
    cal = joblib.load(CALIB_FILE)
    df["alienation_conf_cal"] = cal.predict(df["alienation_conf"].to_numpy())
    return df, True


if records is not None:
    df = build_rows(records)
    df, calibrated = apply_calibration(df)
    try:
        df.to_parquet(OUT_TABLE, index=False)
        written = OUT_TABLE
    except Exception as e:                         # pyarrow missing -> CSV fallback
        written = OUT_TABLE.with_suffix(".csv")
        df.to_csv(written, index=False)
        print(f"parquet unavailable ({e}); wrote CSV instead")
    print(f"wrote {len(df)} rows -> {written.name} | calibrated column: {calibrated}")
else:
    df = None
    print("No records — table not built. Run the scraper/import notebooks first.")

wrote 1116 rows -> echr_extracted.parquet | calibrated column: True


## 7. Report — genre × outcome, alienation balance, confidence distribution

In [ ]:
if df is not None:
    from collections import Counter
    print("genre x outcome:")
    print(df.groupby(["genre", "outcome"], dropna=False).size().to_string())
    print("\noutcome null by genre (must be null for communicated):")
    print(df.assign(is_null=df.outcome.isna()).groupby("genre").is_null.mean().round(3).to_string())

    pos = int(df.alienation_alleged.sum())
    print(f"\nalienation_alleged: {pos}/{len(df)} = {pos/len(df)*100:.1f}% positive")
    print("  positives by genre:", df[df.alienation_alleged].genre.value_counts().to_dict())
    band = pd.cut(df.alienation_conf, [-.01, .05, .25, ALLEGED_THRESHOLD, CONF_HIGH, 1.01],
                  labels=["~0 none", "low", "[.25,.5) reject", "[.5,.7) abstain-zone", ">=.7 confident"])
    print("\nalienation_conf bands (field-level abstention is the mid bands):")
    print(band.value_counts().sort_index().to_string())

genre x outcome:
genre          outcome     
admissibility  admissible        8
               inadmissible    255
               struck out       16
               NaN              32
communicated   NaN             238
merits         no violation    182
               violation       384
               NaN               1

outcome null by genre (must be null for communicated):
genre
admissibility    0.103
communicated     1.000
merits           0.002

alienation_alleged: 54/1116 = 4.8% positive
  positives by genre: {'merits': 34, 'admissibility': 14, 'communicated': 6}

alienation_conf bands (field-level abstention is the mid bands):
alienation_conf
~0 none                 928
low                      93
[.25,.5) reject          41
[.5,.7) abstain-zone     27
>=.7 confident           27


### 7b. Class-balance figure (content field is a minority class) → `figures/`

In [ ]:
if df is not None:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    FIG_DIR = Path("../figures"); FIG_DIR.mkdir(exist_ok=True)
    order = ["merits", "admissibility", "communicated"]
    g = df.groupby("genre")
    rate = (g.alienation_alleged.mean() * 100).reindex(order)
    npos = g.alienation_alleged.sum().reindex(order).astype(int)
    ntot = g.size().reindex(order).astype(int)
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(order, rate.values, color=["#2c7fb8", "#7fcdbb", "#cccccc"])
    for b, p, t in zip(bars, npos.values, ntot.values):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.3, f"{p}/{t}",
                ha="center", va="bottom", fontsize=9)
    overall = df.alienation_alleged.mean() * 100
    ax.axhline(overall, ls="--", c="grey", lw=1)
    ax.text(2.4, overall + 0.3, f"overall {overall:.1f}%", ha="right", color="grey", fontsize=8)
    ax.set_ylabel("% of cases with alienation_alleged=True")
    ax.set_title("ECHR alienation_alleged class balance by genre\n(minority class -> stratified labelling)")
    fig.tight_layout()
    out = FIG_DIR / "echr_alienation_class_balance.png"
    fig.savefig(out, dpi=130); plt.close(fig)
    print("wrote", out.name, "| overall positive rate %.1f%%" % overall)

wrote echr_alienation_class_balance.png | overall positive rate 4.8%


## 8. Stratified labelling template (offline hand-labelling — never used at runtime)
The content field is a minority class (~5% of cases), so a random sample would have too few
positives for a stable F1 and empty confidence bins for the reliability diagram. We
**stratify by the extractor's confidence band**, taking *all* predicted-positive and
abstention-zone cases and sampling the negatives, ~50/50 overall. The template pre-fills the
extractor's guess + evidence + URL and leaves a blank `gold_alienation_alleged` column, so you
label efficiently. **Freeze** the file you label as `data/echr_labeled_sample.csv`.

In [ ]:
if df is not None:
    import numpy as np
    rng = np.random.default_rng(RANDOM_SEED)
    bands = {
        "confident(>=.7)":    df.alienation_conf >= CONF_HIGH,
        "abstain[.5,.7)":     (df.alienation_conf >= ALLEGED_THRESHOLD) & (df.alienation_conf < CONF_HIGH),
        "gray[.25,.5)":       (df.alienation_conf >= 0.25) & (df.alienation_conf < ALLEGED_THRESHOLD),
        "outctx[.05,.25)":    (df.alienation_conf >= 0.05) & (df.alienation_conf < 0.25),
        "none(<.05)":         df.alienation_conf < 0.05,
    }
    # take all of the informative top bands; sample the rest to hit SAMPLE_N (~50% positive-ish)
    targets = {"confident(>=.7)": 999, "abstain[.5,.7)": 999, "gray[.25,.5)": 999,
               "outctx[.05,.25)": 25, "none(<.05)": 40}
    picks = []
    for name, mask in bands.items():
        idx = df.index[mask].to_numpy()
        k = min(targets[name], len(idx))
        take = idx if targets[name] >= len(idx) else rng.choice(idx, size=k, replace=False)
        picks.extend(take.tolist())
    picks = picks[:SAMPLE_N]
    tmpl = df.loc[picks, ["id", "title", "url", "genre", "outcome", "alienation_conf",
                          "alienation_alleged", "alienation_evidence"]].copy()
    tmpl = tmpl.rename(columns={"alienation_alleged": "extractor_guess",
                                "outcome": "outcome_guess"})
    tmpl["gold_alienation_alleged"] = ""          # <-- you fill this (1/0)
    # outcome is metadata-derived (high conf); gold_outcome lets validation measure the one
    # *derived* metadata field. Pre-filled with the guess — usually just confirm, rarely correct.
    tmpl["gold_outcome"] = tmpl["outcome_guess"]
    tmpl["notes"] = ""
    tmpl = tmpl.sample(frac=1, random_state=RANDOM_SEED)   # shuffle so labelling isn't band-ordered
    tmpl.to_csv(LABEL_TEMPLATE, index=False)
    print(f"wrote stratified template: {len(tmpl)} rows -> {LABEL_TEMPLATE.name}")
    print("  band coverage:", {n: int(m.loc[picks].sum()) for n, m in bands.items()})
    print(f"  predicted-positive in sample: {int(tmpl.extractor_guess.sum())} "
          f"({tmpl.extractor_guess.mean()*100:.0f}%)")
    print("\nNEXT: hand-label gold_alienation_alleged (1/0), save as "
          "data/echr_labeled_sample.csv, then run extraction_validation.ipynb (once).")

wrote stratified template: 120 rows -> echr_label_template.csv
  band coverage: {'confident(>=.7)': 27, 'abstain[.5,.7)': 27, 'gray[.25,.5)': 42, 'outctx[.05,.25)': 24, 'none(<.05)': 0}
  predicted-positive in sample: 54 (45%)

NEXT: hand-label gold_alienation_alleged (1/0), save as data/echr_labeled_sample.csv, then run extraction_validation.ipynb (once).


## 9. How this fits the research question
- **Field-level abstention.** `alienation_conf` makes *not knowing* a first-class, queryable
  state: the mid confidence band (≈0.4–0.7) is the honest "this might be an allegation, might
  be the Court's narration" zone. The query layer excludes/flags those cells at a user
  threshold instead of asserting a boolean the rules can't defend.
- **Reusability / generalisation.** The extractor runs **label-free** on every case; gold
  labels enter exactly once, offline, to *calibrate* `raw → P(correct)`. Once fitted, the
  calibration maps confidence on **unseen** ECHR Article 8 cases without any re-labelling —
  the auditable basis for the "labelled once, runs on new data" claim. Provenance on every
  cell keeps the table falsifiable: any value can be traced to the span it came from.